In [1]:
import random

In [2]:
def is_prime(n, k=10):
    if n < 2:
        return False

    small_primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

    for p in small_primes:
        if n == p:
            return True
        if n % p == 0:
            return False

    r = 0
    d = n - 1

    while d % 2 == 0:
        r += 1
        d //= 2

    for _ in range(k):
        a = random.randrange(2, n - 2)
        x = pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        for _ in range(r - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False

    return True

In [3]:
def generate_prime(bits):
    while True:
        num = random.getrandbits(bits)
        num |= (1 << (bits - 1))
        num |= 1

        if is_prime(num):
            return num

In [4]:
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

In [5]:
def generate_keys(bits=512):
    p = generate_prime(bits)
    q = generate_prime(bits)

    while p == q:
        q = generate_prime(bits)

    n = p * q
    phi = (p - 1) * (q - 1)

    e = 65537

    while gcd(e, phi) != 1:
        e += 2

    d = pow(e, -1, phi)

    return (e, n), (d, n)

In [6]:
def encrypt(message, public_key):
    e, n = public_key
    m = int.from_bytes(message.encode(), "big")

    if m >= n:
        raise ValueError("Message too long.")

    return pow(m, e, n)

In [7]:
def decrypt(ciphertext, private_key):
    d, n = private_key
    m = pow(ciphertext, d, n)
    length = (m.bit_length() + 7) // 8
    return m.to_bytes(length, "big").decode()

In [8]:
public_key, private_key = generate_keys(512)

print("Public Key:", public_key)
print("Private Key:", private_key)

Public Key: (65537, 94990948513802638082392408003463714426199009751428033958243635997937542578886893612799250489903225635295956263694638986763715687937619667260521480251297200506226395778768054958586914195080908721974543294949067160578211073283986100335051021971957791984767313686842296868465620127907150635954120075370647599663)
Private Key: (76054214419583624905920234871259716242298464068799789399224256039851972690226178824950825819097640348738108504639899551603913660641512110418451914365107735401035984861936297459225720800310537352248592533633603119517369391830465616013721259041252900069663298770909135228877221432861764572692143244382209842561, 94990948513802638082392408003463714426199009751428033958243635997937542578886893612799250489903225635295956263694638986763715687937619667260521480251297200506226395778768054958586914195080908721974543294949067160578211073283986100335051021971957791984767313686842296868465620127907150635954120075370647599663)


In [9]:
message = "Hello RSA"

ciphertext = encrypt(message, public_key)
print("Ciphertext:", ciphertext)

plaintext = decrypt(ciphertext, private_key)
print("Decrypted:", plaintext)

Ciphertext: 51557015271144467105994847377633338929493312234638777987275298481666273218821072791680218050192803551036639964398701831239889004537242304495171475492580668217811909198707162319765383403709679470026059436427872899262464627418942771288384186937602828802621039454571400557679870026819432823023365304393384211216
Decrypted: Hello RSA
